# A2A Endpoint in LangGraph - Tutorial

This notebook teaches you how to use the Agent-to-Agent (A2A) endpoint in LangGraph. A2A is Google's protocol for enabling communication between conversational AI agents.

## What You'll Learn

1. **Introduction to A2A**: Understanding the A2A protocol and its benefits
2. **Requirements**: Setting up your environment
3. **Creating A2A-Compatible Agents**: Building agents that can communicate via A2A
4. **Agent Card Discovery**: Understanding how agents discover each other
5. **Agent-to-Agent Communication**: Implementing conversations between agents
6. **Best Practices**: Tips for building robust A2A agents

## Prerequisites

- Python 3.10+
- Basic understanding of LangGraph
- OpenAI API key (or another LLM provider)
- `langgraph-api >= 0.4.9` installed


## 1. Introduction to A2A

**Agent-to-Agent (A2A)** is a standardized protocol that enables conversational AI agents to communicate with each other. LangGraph implements A2A support, allowing your agents to:

- **Discover other agents** through agent cards
- **Send messages** using a standardized JSON-RPC format
- **Maintain conversation threads** across agent interactions
- **Handle different message types** (text, structured data, etc.)

The A2A endpoint is available in Agent Server at `/a2a/{assistant_id}`.

### Key Concepts

- **Agent Card**: A JSON document that describes an agent's capabilities and endpoint
- **Message Format**: JSON-RPC 2.0 protocol for sending messages
- **Thread Management**: Conversations are organized into threads
- **State Structure**: Agents must use a message-based state structure


## 2. Requirements and Setup

To use A2A, you need:

1. **langgraph-api >= 0.4.9** - The A2A endpoint is available in this version
2. **Message-based state structure** - Your agent must have a `messages` key in state
3. **Deployed agent** - Either running locally via `langgraph dev` or deployed to production

Let's check if you have the required packages:


In [ ]:
# Check installed packages
import subprocess
import sys

def check_package(package_name, min_version=None):
    """Check if a package is installed and meets version requirements."""
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "show", package_name],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if line.startswith('Version:'):
                    version = line.split(':')[1].strip()
                    print(f"✅ {package_name} is installed (version {version})")
                    if min_version:
                        print(f"   Required: >= {min_version}")
                    return True
        print(f"❌ {package_name} is not installed")
        return False
    except Exception as e:
        print(f"❌ Error checking {package_name}: {e}")
        return False

# Check required packages
print("Checking required packages for A2A:\n")
check_package("langgraph-api", "0.4.9")
check_package("langgraph")
check_package("langchain-openai")
check_package("aiohttp")


## 3. Creating an A2A-Compatible Agent

To create an A2A-compatible agent, you need:

1. **Message-based state**: Your state must include a `messages` key
2. **Proper state structure**: Messages should follow a standard format
3. **Graph definition**: Use LangGraph's StateGraph

Let's create a simple A2A-compatible agent:


In [ ]:
"""Example: Creating an A2A-compatible agent.

This agent processes incoming messages and responds using OpenAI's API.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List
from typing_extensions import TypedDict

from langgraph.graph import StateGraph
from langgraph.runtime import Runtime
from openai import AsyncOpenAI


# Define the context schema (optional configuration parameters)
class Context(TypedDict):
    """Context parameters for the agent."""
    my_configurable_param: str


# Define the state structure
# IMPORTANT: For A2A compatibility, you MUST have a 'messages' key
@dataclass
class State:
    """Input state for the agent.
    
    This defines the initial structure for A2A conversational messages.
    The 'messages' key is required for A2A compatibility.
    """
    messages: List[Dict[str, Any]]


# Define the agent's processing function
async def call_model(state: State, runtime: Runtime[Context]) -> Dict[str, Any]:
    """Process conversational messages and return output using OpenAI."""
    # Initialize OpenAI client
    client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    # Process the incoming messages
    latest_message = state.messages[-1] if state.messages else {}
    user_content = latest_message.get("content", "No message content")

    # Create messages for OpenAI API
    openai_messages = [
        {
            "role": "system",
            "content": "You are a helpful conversational agent. Keep responses brief and engaging."
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    try:
        # Make OpenAI API call
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=openai_messages,
            max_tokens=100,
            temperature=0.7
        )

        ai_response = response.choices[0].message.content

    except Exception as e:
        ai_response = f"I received your message but had trouble processing it. Error: {str(e)[:50]}..."

    # Create a response message
    response_message = {
        "role": "assistant",
        "content": ai_response
    }

    # Return updated state with the new message
    return {
        "messages": state.messages + [response_message]
    }


# Define the graph
graph = (
    StateGraph(State, context_schema=Context)
    .add_node(call_model)
    .add_edge("__start__", "call_model")
    .compile()
)

print("✅ A2A-compatible agent graph created successfully!")
print("\nKey points:")
print("- State includes 'messages' key (required for A2A)")
print("- Messages follow standard format with 'role' and 'content'")
print("- Graph is compiled and ready to use")


In [ ]:
# Example state structure
example_state = State(
    messages=[
        {
            "role": "user",
            "content": "Hello, how are you?"
        },
        {
            "role": "assistant",
            "content": "I'm doing well, thank you! How can I help you today?"
        }
    ]
)

print("Example state structure:")
print(f"Type: {type(example_state)}")
print(f"Messages count: {len(example_state.messages)}")
print("\nMessages:")
for i, msg in enumerate(example_state.messages):
    print(f"  {i+1}. Role: {msg['role']}")
    print(f"     Content: {msg['content']}")


## 4. Agent Card Discovery

Each assistant automatically exposes an **Agent Card** that describes its capabilities. The agent card includes:

- Assistant name and description
- Available skills
- Supported input/output modes
- A2A endpoint URL for communication

You can retrieve the agent card using:

```
GET /.well-known/agent-card.json?assistant_id={assistant_id}
```

Let's see how to retrieve an agent card:


In [ ]:
import aiohttp
import asyncio

async def get_agent_card(port: int, assistant_id: str):
    """Retrieve the agent card for a given assistant."""
    url = f"http://127.0.0.1:{port}/.well-known/agent-card.json"
    params = {"assistant_id": assistant_id}
    
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(url, params=params) as response:
                if response.status == 200:
                    card = await response.json()
                    return card
                else:
                    print(f"Error: {response.status} - {await response.text()}")
                    return None
        except aiohttp.ClientError as e:
            print(f"Connection error: {e}")
            print("\n💡 Make sure your agent server is running with:")
            print(f"   langgraph dev --port {port}")
            return None

# Example usage (uncomment when you have a running agent)
# assistant_id = "your-assistant-id-here"
# card = await get_agent_card(2024, assistant_id)
# if card:
#     print("Agent Card:")
#     print(f"  Name: {card.get('name', 'N/A')}")
#     print(f"  Description: {card.get('description', 'N/A')}")
#     print(f"  Endpoint: {card.get('a2a_endpoint', 'N/A')}")

print("✅ Agent card retrieval function ready!")
print("\nTo use this function:")
print("1. Start your agent: langgraph dev --port 2024")
print("2. Get the assistant_id from the output")
print("3. Call: card = await get_agent_card(2024, assistant_id)")


In [ ]:
async def send_a2a_message(
    session: aiohttp.ClientSession,
    port: int,
    assistant_id: str,
    text: str,
    thread_id: str = ""
) -> str:
    """
    Send a message to an agent via the A2A endpoint and return the response text.
    
    Args:
        session: aiohttp ClientSession
        port: Port number where the agent server is running
        assistant_id: The assistant ID to send the message to
        text: The message text to send
        thread_id: Optional thread ID for conversation continuity
    
    Returns:
        The response text from the agent
    """
    url = f"http://127.0.0.1:{port}/a2a/{assistant_id}"
    
    # A2A protocol JSON-RPC 2.0 payload
    payload = {
        "jsonrpc": "2.0",
        "id": "",
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": text}]
            },
            "messageId": "",
            "thread": {"threadId": thread_id}
        }
    }
    
    headers = {"Accept": "application/json"}
    
    try:
        async with session.post(url, json=payload, headers=headers) as response:
            if response.status == 200:
                result = await response.json()
                # Extract the response text from the A2A response structure
                return result["result"]["artifacts"][0]["parts"][0]["text"]
            else:
                error_text = await response.text()
                print(f"❌ Error {response.status}: {error_text}")
                return f"Error: {response.status}"
    except Exception as e:
        print(f"❌ Exception: {e}")
        return f"Exception: {str(e)}"

print("✅ A2A message sending function created!")
print("\nFunction signature:")
print("  send_a2a_message(session, port, assistant_id, text, thread_id='')")


In [ ]:
async def example_single_message():
    """Example of sending a single message to an agent."""
    # Replace with your actual assistant ID
    assistant_id = os.getenv("AGENT_A_ID", "your-assistant-id-here")
    port = 2024
    
    if assistant_id == "your-assistant-id-here":
        print("⚠️  Set AGENT_A_ID environment variable or update the code")
        print("   Example: assistant_id = 'your-actual-assistant-id'")
        return
    
    async with aiohttp.ClientSession() as session:
        message = "Hello! Can you tell me a fun fact?"
        print(f"📤 Sending: {message}")
        
        response = await send_a2a_message(session, port, assistant_id, message)
        print(f"📥 Response: {response}")

# Uncomment to run (requires a running agent server)
# await example_single_message()

print("✅ Single message example function ready!")
print("\nTo use:")
print("1. Start agent: langgraph dev --port 2024")
print("2. Set AGENT_A_ID environment variable")
print("3. Run: await example_single_message()")


### Example: Multi-Agent Conversation

Now let's create a more complex example where two agents have a conversation:


In [ ]:
async def simulate_agent_conversation(
    agent_a_port: int,
    agent_a_id: str,
    agent_b_port: int,
    agent_b_id: str,
    initial_message: str = "Hello! Let's have a conversation.",
    num_rounds: int = 3
):
    """
    Simulate a conversation between two agents.
    
    Args:
        agent_a_port: Port for agent A
        agent_a_id: Assistant ID for agent A
        agent_b_port: Port for agent B
        agent_b_id: Assistant ID for agent B
        initial_message: First message to start the conversation
        num_rounds: Number of conversation rounds
    """
    message = initial_message
    
    async with aiohttp.ClientSession() as session:
        for i in range(num_rounds):
            print(f"\n{'='*50}")
            print(f"Round {i + 1}")
            print(f"{'='*50}")
            
            # Agent A responds
            print(f"\n🔵 Agent A (port {agent_a_port}):")
            print(f"   📤 Receives: {message}")
            message = await send_a2a_message(session, agent_a_port, agent_a_id, message)
            print(f"   📥 Responds: {message}")
            
            # Agent B responds
            print(f"\n🔴 Agent B (port {agent_b_port}):")
            print(f"   📤 Receives: {message}")
            message = await send_a2a_message(session, agent_b_port, agent_b_id, message)
            print(f"   📥 Responds: {message}")
    
    print(f"\n{'='*50}")
    print("Conversation complete!")
    print(f"{'='*50}")

# Example usage (uncomment when you have two running agents)
# agent_a_id = os.getenv("AGENT_A_ID")
# agent_b_id = os.getenv("AGENT_B_ID")
# 
# if agent_a_id and agent_b_id:
#     await simulate_agent_conversation(
#         agent_a_port=2024,
#         agent_a_id=agent_a_id,
#         agent_b_port=2025,
#         agent_b_id=agent_b_id,
#         initial_message="Hello! Let's discuss artificial intelligence.",
#         num_rounds=3
#     )
# else:
#     print("⚠️  Set AGENT_A_ID and AGENT_B_ID environment variables")

print("✅ Multi-agent conversation function ready!")
print("\nTo use:")
print("1. Start agent A: langgraph dev --port 2024")
print("2. Start agent B: langgraph dev --port 2025")
print("3. Set AGENT_A_ID and AGENT_B_ID environment variables")
print("4. Run: await simulate_agent_conversation(...)")


In [ ]:
# Complete example: Full workflow
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

async def complete_example():
    """
    Complete example demonstrating the full A2A workflow.
    
    Steps:
    1. Check if agents are configured
    2. Retrieve agent cards (optional)
    3. Send messages between agents
    """
    agent_a_id = os.getenv("AGENT_A_ID")
    agent_b_id = os.getenv("AGENT_B_ID")
    
    if not agent_a_id or not agent_b_id:
        print("⚠️  Configuration required:")
        print("   1. Start agent A: langgraph dev --port 2024")
        print("   2. Start agent B: langgraph dev --port 2025")
        print("   3. Set environment variables:")
        print("      export AGENT_A_ID='<id-from-port-2024>'")
        print("      export AGENT_B_ID='<id-from-port-2025>'")
        print("\n   Or create a .env file with:")
        print("      AGENT_A_ID=<id-from-port-2024>")
        print("      AGENT_B_ID=<id-from-port-2025>")
        return
    
    print("🚀 Starting A2A conversation example...\n")
    
    # Step 1: Retrieve agent cards (optional but useful)
    print("Step 1: Retrieving agent cards...")
    async with aiohttp.ClientSession() as session:
        card_a = await get_agent_card(2024, agent_a_id)
        card_b = await get_agent_card(2025, agent_b_id)
        
        if card_a:
            print(f"✅ Agent A: {card_a.get('name', 'Unknown')}")
        if card_b:
            print(f"✅ Agent B: {card_b.get('name', 'Unknown')}")
    
    # Step 2: Start conversation
    print("\nStep 2: Starting conversation...")
    await simulate_agent_conversation(
        agent_a_port=2024,
        agent_a_id=agent_a_id,
        agent_b_port=2025,
        agent_b_id=agent_b_id,
        initial_message="Hi! I'm interested in learning about LangGraph. Can you help?",
        num_rounds=3
    )

# Uncomment to run the complete example
# await complete_example()

print("✅ Complete example function ready!")
print("\nThis function demonstrates:")
print("  1. Environment variable configuration")
print("  2. Agent card retrieval")
print("  3. Multi-agent conversation")


## 7. Best Practices

Here are some best practices for building A2A-compatible agents:

### 1. State Structure
- ✅ Always include a `messages` key in your state
- ✅ Use consistent message format: `{"role": "...", "content": "..."}`
- ✅ Maintain conversation history in the messages array

### 2. Error Handling
- ✅ Handle API failures gracefully
- ✅ Return meaningful error messages
- ✅ Log errors for debugging

### 3. Thread Management
- ✅ Use thread IDs for conversation continuity
- ✅ Support both new and existing threads
- ✅ Track thread state appropriately

### 4. Message Format
- ✅ Follow JSON-RPC 2.0 protocol
- ✅ Include all required fields in the payload
- ✅ Handle different message parts (text, structured data, etc.)

### 5. Testing
- ✅ Test with local agents first (`langgraph dev`)
- ✅ Verify agent cards are accessible
- ✅ Test error scenarios (offline agents, invalid IDs, etc.)

### 6. Security
- ✅ Validate incoming messages
- ✅ Implement authentication if needed
- ✅ Sanitize user inputs

Let's create a helper function with better error handling:


In [ ]:
async def send_a2a_message_robust(
    session: aiohttp.ClientSession,
    port: int,
    assistant_id: str,
    text: str,
    thread_id: str = "",
    timeout: int = 30
) -> dict:
    """
    Robust version of send_a2a_message with better error handling.
    
    Returns:
        dict with keys: 'success', 'response', 'error'
    """
    url = f"http://127.0.0.1:{port}/a2a/{assistant_id}"
    
    payload = {
        "jsonrpc": "2.0",
        "id": "",
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": text}]
            },
            "messageId": "",
            "thread": {"threadId": thread_id}
        }
    }
    
    headers = {"Accept": "application/json"}
    timeout_obj = aiohttp.ClientTimeout(total=timeout)
    
    try:
        async with session.post(url, json=payload, headers=headers, timeout=timeout_obj) as response:
            if response.status == 200:
                result = await response.json()
                response_text = result["result"]["artifacts"][0]["parts"][0]["text"]
                return {
                    "success": True,
                    "response": response_text,
                    "error": None
                }
            else:
                error_text = await response.text()
                return {
                    "success": False,
                    "response": None,
                    "error": f"HTTP {response.status}: {error_text}"
                }
    except asyncio.TimeoutError:
        return {
            "success": False,
            "response": None,
            "error": f"Timeout after {timeout} seconds"
        }
    except aiohttp.ClientError as e:
        return {
            "success": False,
            "response": None,
            "error": f"Connection error: {str(e)}"
        }
    except Exception as e:
        return {
            "success": False,
            "response": None,
            "error": f"Unexpected error: {str(e)}"
        }

# Example usage
async def example_robust_message():
    """Example using the robust message sending function."""
    assistant_id = os.getenv("AGENT_A_ID", "test-id")
    
    async with aiohttp.ClientSession() as session:
        result = await send_a2a_message_robust(
            session, 2024, assistant_id, "Hello!", timeout=10
        )
        
        if result["success"]:
            print(f"✅ Success: {result['response']}")
        else:
            print(f"❌ Error: {result['error']}")

print("✅ Robust message sending function created!")
print("\nFeatures:")
print("  - Better error handling")
print("  - Timeout support")
print("  - Structured return values")
print("  - Connection error handling")


In [ ]:
async def diagnose_agent(port: int, assistant_id: str):
    """
    Diagnose an agent's A2A endpoint.
    
    Checks:
    1. Server connectivity
    2. Agent card availability
    3. A2A endpoint accessibility
    """
    print(f"🔍 Diagnosing agent on port {port} (ID: {assistant_id})\n")
    
    async with aiohttp.ClientSession() as session:
        # Check 1: Server connectivity
        print("1. Checking server connectivity...")
        try:
            async with session.get(f"http://127.0.0.1:{port}/", timeout=aiohttp.ClientTimeout(total=5)) as response:
                print(f"   ✅ Server is reachable (status: {response.status})")
        except Exception as e:
            print(f"   ❌ Server not reachable: {e}")
            print(f"   💡 Make sure to run: langgraph dev --port {port}")
            return
        
        # Check 2: Agent card
        print("\n2. Checking agent card...")
        card = await get_agent_card(port, assistant_id)
        if card:
            print(f"   ✅ Agent card retrieved")
            print(f"      Name: {card.get('name', 'N/A')}")
            print(f"      Description: {card.get('description', 'N/A')[:50]}...")
        else:
            print(f"   ❌ Could not retrieve agent card")
            print(f"   💡 Check that the assistant_id is correct")
        
        # Check 3: A2A endpoint
        print("\n3. Testing A2A endpoint...")
        result = await send_a2a_message_robust(
            session, port, assistant_id, "Test message", timeout=10
        )
        if result["success"]:
            print(f"   ✅ A2A endpoint working")
            print(f"      Response: {result['response'][:50]}...")
        else:
            print(f"   ❌ A2A endpoint error: {result['error']}")
            print(f"   💡 Check agent state structure includes 'messages' key")
    
    print("\n✅ Diagnosis complete!")

# Example usage
# await diagnose_agent(2024, "your-assistant-id")

print("✅ Diagnostic function created!")
print("\nUse this to troubleshoot agent connectivity issues.")


## 9. Summary

In this tutorial, you learned:

1. ✅ **What A2A is**: A protocol for agent-to-agent communication
2. ✅ **Requirements**: langgraph-api >= 0.4.9 and message-based state
3. ✅ **Creating A2A agents**: State must include `messages` key
4. ✅ **Agent cards**: How agents discover each other
5. ✅ **Sending messages**: JSON-RPC 2.0 format
6. ✅ **Multi-agent conversations**: Coordinating multiple agents
7. ✅ **Best practices**: Error handling, security, testing
8. ✅ **Troubleshooting**: Common issues and solutions

## Next Steps

1. **Deploy your agent**: Use `langgraph dev` for local testing
2. **Test with multiple agents**: Set up two or more agents
3. **Explore advanced features**: Thread management, structured data
4. **Read the documentation**: https://docs.langchain.com/langsmith/server-a2a

## Resources

- [LangGraph A2A Documentation](https://docs.langchain.com/langsmith/server-a2a#a2a-endpoint-in-agent-server)
- [JSON-RPC 2.0 Specification](https://www.jsonrpc.org/specification)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)

Happy building! 🚀
